[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/study_notes/week02_real_world_analysis.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 2: Real-World A/B Analysis & Metric Computation

**Course**: Advanced A/B Testing | **Context**: QuickCart (online electronics retailer)

**Your role**: You've validated your statistical foundations. Now you're looking at *real* experiment data for the first time. The data is messy, metrics behave unexpectedly, and you need to understand what can go wrong before trusting any result.

---

## What You'll Learn

1. Computing experiment metrics from raw event-level data
2. Best practices for metric definition (what makes a good experiment metric)
3. Daily metric aggregation and time-series monitoring
4. Practical issues: seasonality, novelty effects, day-of-week patterns
5. Metric sensitivity analysis (which metrics are easier to move?)
6. Building a metric monitoring dashboard for experiments
7. Anomaly detection in metric time series

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import scipy.stats as stats
from datetime import datetime, timedelta

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---

## 1. From Raw Events to Experiment Metrics

### The Problem

QuickCart's event log contains millions of rows: page views, clicks, add-to-carts, purchases, refunds. Your experiment (a new checkout flow) has been running for 28 days. But raw events are not metrics -- you need to **aggregate** them into something meaningful.

The general pattern:
1. **Filter** events to the experiment period and relevant population
2. **Aggregate** to the analysis unit (usually per-user or per-day)
3. **Compute** the metric value

### Generating Synthetic Experiment Data

We create a realistic QuickCart sales dataset with known ground truth, so we can verify our metric computations.

In [ ]:
def generate_quickcart_data(n_days=28, n_users=5000, treatment_effect=0.05, seed=42):
    """
    Generate synthetic QuickCart experiment data.
    
    Simulates a 28-day experiment with:
    - Day-of-week seasonality (weekends have higher revenue)
    - A novelty effect that decays over the first week
    - Random daily variation
    - A true treatment effect on revenue per user
    
    Returns a DataFrame with columns:
        sale_id, user_id, group, day, cost, quantity, category
    """
    rng = np.random.default_rng(seed)
    
    start_date = pd.Timestamp('2024-03-01')
    dates = [start_date + timedelta(days=i) for i in range(n_days)]
    
    records = []
    sale_id = 0
    
    # Assign users to groups
    user_groups = {uid: 'control' if uid < n_users // 2 else 'treatment' 
                   for uid in range(n_users)}
    
    categories = ['phones', 'laptops', 'accessories', 'audio', 'gaming']
    base_prices = {'phones': 600, 'laptops': 900, 'accessories': 35, 
                   'audio': 120, 'gaming': 250}
    
    for day_idx, date in enumerate(dates):
        dow = date.dayofweek  # 0=Mon, 6=Sun
        
        # Seasonality: weekends have 30% more transactions
        weekend_multiplier = 1.3 if dow >= 5 else 1.0
        
        # Novelty effect: treatment has +15% extra lift in first 3 days, decaying
        novelty_factor = 0.15 * max(0, 1 - day_idx / 3)
        
        # Daily random variation
        daily_noise = rng.normal(1.0, 0.05)
        
        # Each user has a probability of purchasing on this day
        base_purchase_prob = 0.08 * weekend_multiplier * daily_noise
        
        for uid in range(n_users):
            group = user_groups[uid]
            
            # Treatment effect on purchase probability
            if group == 'treatment':
                prob = base_purchase_prob * (1 + treatment_effect + novelty_factor)
            else:
                prob = base_purchase_prob
            
            if rng.random() < prob:
                category = rng.choice(categories, p=[0.25, 0.15, 0.35, 0.15, 0.10])
                base_price = base_prices[category]
                # Price variation
                cost = max(5, rng.normal(base_price, base_price * 0.3))
                quantity = rng.choice([1, 1, 1, 2, 3], p=[0.6, 0.15, 0.1, 0.1, 0.05])
                
                records.append({
                    'sale_id': sale_id,
                    'user_id': uid,
                    'group': group,
                    'day': date,
                    'cost': round(cost * quantity, 2),
                    'quantity': quantity,
                    'category': category
                })
                sale_id += 1
    
    return pd.DataFrame(records)


# Generate the data
df = generate_quickcart_data()
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['day'].min()} to {df['day'].max()}")
print(f"Groups: {df['group'].value_counts().to_dict()}")
print(f"\nSample records:")
df.head(10)

---

## 2. The `calculate_sales_metrics` Function

This is the core pattern for computing experiment metrics from raw data. The function:
1. Applies optional filters (date range, category, user segment)
2. Aggregates to daily level
3. Computes summary statistics over a specified period

This mirrors the approach from the homework, where we need flexible metric computation across different slices of data.

In [ ]:
def calculate_sales_metrics(df, cost_name='cost', date_name='day', 
                            sale_id_name='sale_id', period=7, filters=None):
    """
    Compute daily aggregate sales metrics from event-level data.
    
    Parameters
    ----------
    df : pd.DataFrame
        Raw event-level sales data.
    cost_name : str
        Column name for the monetary value (revenue/cost).
    date_name : str
        Column name for the date/day.
    sale_id_name : str
        Column name for the transaction identifier.
    period : int
        Number of days to aggregate over (takes the last N days).
    filters : dict or None
        Column-value pairs to filter the data before aggregation.
        Example: {'group': 'treatment', 'category': 'phones'}
    
    Returns
    -------
    dict with keys:
        'daily_revenue': array of daily total revenue
        'daily_transactions': array of daily transaction counts
        'daily_aov': array of daily average order values
        'total_revenue': sum over period
        'total_transactions': count over period
        'avg_daily_revenue': mean daily revenue
        'avg_aov': mean average order value
        'dates': array of dates in the period
    """
    data = df.copy()
    
    # Apply filters
    if filters:
        for col, val in filters.items():
            if col in data.columns:
                data = data[data[col] == val]
    
    # Sort by date and take the last N days
    all_dates = sorted(data[date_name].unique())
    if len(all_dates) > period:
        period_dates = all_dates[-period:]
        data = data[data[date_name].isin(period_dates)]
    else:
        period_dates = all_dates
    
    # Daily aggregation
    daily = data.groupby(date_name).agg(
        daily_revenue=(cost_name, 'sum'),
        daily_transactions=(sale_id_name, 'count'),
    ).reset_index()
    daily['daily_aov'] = daily['daily_revenue'] / daily['daily_transactions']
    
    return {
        'daily_revenue': daily['daily_revenue'].values,
        'daily_transactions': daily['daily_transactions'].values,
        'daily_aov': daily['daily_aov'].values,
        'total_revenue': daily['daily_revenue'].sum(),
        'total_transactions': daily['daily_transactions'].sum(),
        'avg_daily_revenue': daily['daily_revenue'].mean(),
        'avg_aov': daily['daily_aov'].mean(),
        'dates': daily[date_name].values
    }


# Compute metrics for control and treatment
control_metrics = calculate_sales_metrics(df, period=14, filters={'group': 'control'})
treatment_metrics = calculate_sales_metrics(df, period=14, filters={'group': 'treatment'})

print("=== Control Group (last 14 days) ===")
print(f"  Total revenue:        ${control_metrics['total_revenue']:,.0f}")
print(f"  Total transactions:   {control_metrics['total_transactions']:,}")
print(f"  Avg daily revenue:    ${control_metrics['avg_daily_revenue']:,.0f}")
print(f"  Avg order value:      ${control_metrics['avg_aov']:.2f}")

print(f"\n=== Treatment Group (last 14 days) ===")
print(f"  Total revenue:        ${treatment_metrics['total_revenue']:,.0f}")
print(f"  Total transactions:   {treatment_metrics['total_transactions']:,}")
print(f"  Avg daily revenue:    ${treatment_metrics['avg_daily_revenue']:,.0f}")
print(f"  Avg order value:      ${treatment_metrics['avg_aov']:.2f}")

# Relative lift
revenue_lift = (treatment_metrics['avg_daily_revenue'] - control_metrics['avg_daily_revenue']) / control_metrics['avg_daily_revenue']
print(f"\n  Revenue lift: {revenue_lift:+.2%}")

---

## 3. What Makes a Good Experiment Metric?

Not all metrics are created equal. A good experiment metric should be:

| Property | Why it matters | Example (good) | Example (bad) |
|----------|---------------|----------------|---------------|
| **Sensitive** | Detects real effects quickly | Revenue per user | Total company revenue |
| **Directional** | Clear which direction is better | Conversion rate (higher=better) | "Engagement" (ambiguous) |
| **Attributable** | Can be tied to treatment | Per-user revenue in experiment | Stock price |
| **Timely** | Observable within experiment duration | Purchase within 7 days | Lifetime value |
| **Robust** | Not dominated by outliers | Median order value | Mean revenue (unbounded) |

### QuickCart's Metric Hierarchy

At QuickCart, we define three levels of metrics:

1. **Primary metric** (Overall Evaluation Criterion): Revenue per user -- this is what we make ship/no-ship decisions on
2. **Secondary metrics**: Conversion rate, average order value, transactions per user
3. **Guardrail metrics**: Page load time, error rate, refund rate -- must not degrade

:::{admonition} The Metric Definition Trap
:class: warning
A common mistake is defining metrics *after* seeing results. This leads to cherry-picking -- you'll always find some metric that moved. Define your metrics and success criteria **before** the experiment starts.
:::

In [ ]:
# Compute multiple metrics to demonstrate the hierarchy
def compute_user_level_metrics(df, n_users=5000):
    """Compute per-user metrics for statistical testing."""
    # Get all users (including those with zero purchases)
    all_users = pd.DataFrame({'user_id': range(n_users)})
    all_users['group'] = all_users['user_id'].apply(
        lambda x: 'control' if x < n_users // 2 else 'treatment'
    )
    
    # Per-user aggregation
    user_stats = df.groupby(['user_id', 'group']).agg(
        revenue=('cost', 'sum'),
        transactions=('sale_id', 'count'),
        avg_order_value=('cost', 'mean')
    ).reset_index()
    
    # Merge to include zero-purchase users
    user_metrics = all_users.merge(user_stats, on=['user_id', 'group'], how='left')
    user_metrics = user_metrics.fillna(0)
    user_metrics['converted'] = (user_metrics['transactions'] > 0).astype(int)
    
    return user_metrics


user_metrics = compute_user_level_metrics(df)

# Statistical tests on each metric
control_users = user_metrics[user_metrics['group'] == 'control']
treatment_users = user_metrics[user_metrics['group'] == 'treatment']

metrics_to_test = ['revenue', 'transactions', 'converted', 'avg_order_value']
print("Metric-level results (Welch's t-test):")
print("-" * 70)
print(f"{'Metric':<20} {'Control Mean':>12} {'Treatment Mean':>14} {'Lift':>8} {'p-value':>10}")
print("-" * 70)

for metric in metrics_to_test:
    c = control_users[metric].values
    t = treatment_users[metric].values
    stat, pval = stats.ttest_ind(c, t, equal_var=False)
    lift = (t.mean() - c.mean()) / c.mean() if c.mean() != 0 else 0
    sig = '*' if pval < 0.05 else ''
    print(f"{metric:<20} {c.mean():>12.4f} {t.mean():>14.4f} {lift:>+7.2%} {pval:>9.4f} {sig}")

---

## 4. Daily Metric Aggregation and Time-Series Monitoring

Looking at a single number ("treatment is +5%") hides critical information. We need to track metrics **over time** to spot:
- Is the effect stable or growing/shrinking?
- Are there anomalous days that distort the aggregate?
- Is there a day-of-week pattern we need to account for?

In [ ]:
# Compute daily metrics for both groups
n_control_users = 2500
n_treatment_users = 2500

daily_control = df[df['group'] == 'control'].groupby('day').agg(
    revenue=('cost', 'sum'),
    transactions=('sale_id', 'count'),
    unique_users=('user_id', 'nunique')
).reset_index()
daily_control['revenue_per_user'] = daily_control['revenue'] / n_control_users
daily_control['aov'] = daily_control['revenue'] / daily_control['transactions']

daily_treatment = df[df['group'] == 'treatment'].groupby('day').agg(
    revenue=('cost', 'sum'),
    transactions=('sale_id', 'count'),
    unique_users=('user_id', 'nunique')
).reset_index()
daily_treatment['revenue_per_user'] = daily_treatment['revenue'] / n_treatment_users
daily_treatment['aov'] = daily_treatment['revenue'] / daily_treatment['transactions']

# Plot daily revenue per user
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

axes[0].plot(daily_control['day'], daily_control['revenue_per_user'], 
             'b-o', label='Control', markersize=4)
axes[0].plot(daily_treatment['day'], daily_treatment['revenue_per_user'], 
             'r-o', label='Treatment', markersize=4)
axes[0].set_ylabel('Revenue per user ($)')
axes[0].set_title('Daily Revenue per User: Control vs Treatment')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Highlight weekends
for day in daily_control['day']:
    if pd.Timestamp(day).dayofweek >= 5:
        axes[0].axvspan(day - timedelta(hours=12), day + timedelta(hours=12), 
                       alpha=0.05, color='gray')

# Plot the daily lift (relative difference)
daily_lift = ((daily_treatment['revenue_per_user'].values - 
               daily_control['revenue_per_user'].values) / 
              daily_control['revenue_per_user'].values)

colors = ['green' if x > 0 else 'red' for x in daily_lift]
axes[1].bar(daily_control['day'], daily_lift * 100, color=colors, alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axhline(5, color='green', linestyle='--', alpha=0.5, label='True effect (5%)')
axes[1].set_ylabel('Daily lift (%)')
axes[1].set_xlabel('Date')
axes[1].set_title('Daily Treatment Lift (Revenue per User)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Note: The true treatment effect is 5%, but daily estimates are noisy.")
print(f"Daily lift range: [{daily_lift.min()*100:.1f}%, {daily_lift.max()*100:.1f}%]")
print(f"Average daily lift: {daily_lift.mean()*100:.1f}%")

---

## 5. Practical Issues: Seasonality, Novelty, and Day-of-Week

### Day-of-Week Patterns

E-commerce traffic and purchasing patterns vary systematically by day of week. If your experiment starts on a Monday and you check results on Wednesday, you've only seen weekdays -- your metric estimate is biased relative to a full-week view.

**Best practice**: Always analyze in complete weeks (7, 14, 21, 28 days).

In [ ]:
# Day-of-week analysis
daily_all = df.groupby('day').agg(
    revenue=('cost', 'sum'),
    transactions=('sale_id', 'count')
).reset_index()
daily_all['dow'] = pd.to_datetime(daily_all['day']).dt.day_name()
daily_all['dow_num'] = pd.to_datetime(daily_all['day']).dt.dayofweek

# Average by day of week
dow_avg = daily_all.groupby(['dow', 'dow_num']).agg(
    avg_revenue=('revenue', 'mean'),
    avg_transactions=('transactions', 'mean')
).reset_index().sort_values('dow_num')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3'] * 5 + ['#FF9800'] * 2  # blue weekdays, orange weekends
axes[0].bar(dow_avg['dow'], dow_avg['avg_revenue'], color=colors, alpha=0.8)
axes[0].set_title('Average Daily Revenue by Day of Week')
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(dow_avg['dow'], dow_avg['avg_transactions'], color=colors, alpha=0.8)
axes[1].set_title('Average Daily Transactions by Day of Week')
axes[1].set_ylabel('Transactions')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

weekend_revenue = dow_avg[dow_avg['dow_num'] >= 5]['avg_revenue'].mean()
weekday_revenue = dow_avg[dow_avg['dow_num'] < 5]['avg_revenue'].mean()
print(f"Weekend vs weekday revenue ratio: {weekend_revenue/weekday_revenue:.2f}x")

### Novelty Effect

When users first encounter a change, they may engage more (curiosity) or less (confusion). This "novelty effect" inflates or deflates the measured treatment effect in the early days.

Our synthetic data includes a decaying novelty effect in the first 3 days. Let's see if we can detect it.

In [ ]:
# Cumulative lift over time (how does our estimate evolve?)
dates = sorted(df['day'].unique())
cumulative_lifts = []

for i in range(1, len(dates) + 1):
    subset = df[df['day'].isin(dates[:i])]
    c_rev = subset[subset['group'] == 'control']['cost'].sum() / n_control_users
    t_rev = subset[subset['group'] == 'treatment']['cost'].sum() / n_treatment_users
    lift = (t_rev - c_rev) / c_rev if c_rev > 0 else 0
    cumulative_lifts.append(lift)

plt.figure(figsize=(12, 5))
plt.plot(dates, np.array(cumulative_lifts) * 100, 'b-o', markersize=4)
plt.axhline(5, color='green', linestyle='--', label='True long-run effect (5%)', alpha=0.7)
plt.axhspan(0, 5, alpha=0.05, color='green')
plt.axvline(dates[3], color='red', linestyle=':', alpha=0.7, label='Novelty period ends')
plt.xlabel('Date')
plt.ylabel('Cumulative lift (%)')
plt.title('Cumulative Treatment Effect Over Time (Novelty Decay Visible)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Lift after 3 days: {cumulative_lifts[2]*100:.1f}% (inflated by novelty)")
print(f"Lift after 7 days: {cumulative_lifts[6]*100:.1f}%")
print(f"Lift after 14 days: {cumulative_lifts[13]*100:.1f}%")
print(f"Lift after 28 days: {cumulative_lifts[-1]*100:.1f}% (closest to true effect)")

:::{admonition} How to Handle Novelty Effects
:class: dropdown

**Detection**:
- Plot cumulative lift over time (as above)
- Compare first-week lift to subsequent weeks
- Look for a clear decreasing trend in daily lifts

**Mitigation**:
1. **Exclude the burn-in period**: Drop the first 3-7 days from your analysis
2. **Compare weekly cohorts**: Week 1 users vs Week 2+ users
3. **Run longer experiments**: 2-4 weeks minimum to let novelty wear off
4. **Look at returning users**: New-to-treatment users show novelty; returning users show the steady-state effect

**At QuickCart**, we require a minimum 14-day experiment duration and always check the time-series plot before declaring results.
:::

---

## 6. Metric Sensitivity Analysis

### Which Metrics Are Easier to Move?

A metric's **sensitivity** determines how quickly an experiment can detect a real effect. Sensitivity depends on:

$$
\text{Sensitivity} \propto \frac{\text{Effect Size}}{\text{Standard Deviation} / \sqrt{n}}
$$

The coefficient of variation (CV = std/mean) is a useful proxy: metrics with lower CV are more sensitive because the signal-to-noise ratio is better.

Let's compare the sensitivity of different QuickCart metrics.

In [ ]:
# Compute per-user metrics and their statistical properties
user_metrics = compute_user_level_metrics(df)

# Focus on users who made at least one purchase for AOV
purchasers = user_metrics[user_metrics['converted'] == 1]

metrics_sensitivity = {
    'Revenue per user (all)': user_metrics['revenue'],
    'Transactions per user (all)': user_metrics['transactions'],
    'Conversion rate': user_metrics['converted'],
    'Revenue per buyer': purchasers['revenue'],
    'AOV (buyers only)': purchasers['avg_order_value'],
}

print("Metric Sensitivity Analysis")
print("=" * 75)
print(f"{'Metric':<30} {'Mean':>10} {'Std':>10} {'CV':>8} {'MDE (n=2500)':>12}")
print("-" * 75)

sensitivity_data = []
for name, values in metrics_sensitivity.items():
    mean_val = values.mean()
    std_val = values.std()
    cv = std_val / mean_val if mean_val > 0 else float('inf')
    # MDE detectable with n=2500 per group, alpha=0.05, power=0.80
    # MDE ~ 2.8 * std / sqrt(n) / mean (relative)
    n = len(values) // 2
    mde_relative = 2.8 * std_val / np.sqrt(n) / mean_val if mean_val > 0 else float('inf')
    sensitivity_data.append({'name': name, 'cv': cv, 'mde': mde_relative})
    print(f"{name:<30} {mean_val:>10.2f} {std_val:>10.2f} {cv:>8.2f} {mde_relative:>11.1%}")

print("\nInterpretation: 'MDE' = minimum detectable effect (lower = more sensitive)")
print("Conversion rate is the most sensitive metric (lowest MDE).")

In [ ]:
# Visualize: MDE comparison
names = [d['name'] for d in sensitivity_data]
mdes = [d['mde'] * 100 for d in sensitivity_data]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, mdes, color='steelblue', alpha=0.8)
ax.set_xlabel('Minimum Detectable Effect (%)')
ax.set_title('Metric Sensitivity: Lower MDE = Easier to Detect Changes')
ax.grid(alpha=0.3, axis='x')

# Add value labels
for bar, mde in zip(bars, mdes):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, 
            f'{mde:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

:::{admonition} Metric Sensitivity -- Key Insight
:class: tip

**Binary metrics** (like conversion rate) tend to have lower variance relative to their mean, making them more sensitive. **Revenue metrics** are high-variance because of the long tail in order values.

This is why many teams use conversion rate as their primary metric for decision-making, even though revenue is what ultimately matters for the business. The trade-off:
- Conversion detects effects faster (needs fewer users)
- Revenue captures the full economic impact (including order size changes)

Advanced techniques (CUPED, stratification) in later weeks will reduce variance in revenue metrics, making them more practical as primary metrics.
:::

---

## 7. Building a Metric Monitoring Dashboard

A good experiment dashboard shows:
1. **Time series** of the metric for both groups
2. **Cumulative effect** with confidence interval
3. **Sample ratio mismatch** check (are groups balanced?)
4. **Guardrail metrics** status

Let's build a simplified version.

In [ ]:
def experiment_dashboard(df, metric_col='cost', user_col='user_id',
                         group_col='group', date_col='day',
                         n_control=2500, n_treatment=2500):
    """
    Generate a 4-panel experiment monitoring dashboard.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # --- Panel 1: Daily metric time series ---
    daily_c = df[df[group_col] == 'control'].groupby(date_col)[metric_col].sum() / n_control
    daily_t = df[df[group_col] == 'treatment'].groupby(date_col)[metric_col].sum() / n_treatment
    
    axes[0, 0].plot(daily_c.index, daily_c.values, 'b-o', markersize=3, label='Control')
    axes[0, 0].plot(daily_t.index, daily_t.values, 'r-o', markersize=3, label='Treatment')
    axes[0, 0].set_title('Daily Revenue per User')
    axes[0, 0].set_ylabel('$ per user')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # --- Panel 2: Cumulative lift with CI ---
    dates_sorted = sorted(df[date_col].unique())
    cum_lifts = []
    cum_ci_low = []
    cum_ci_high = []
    
    for i in range(1, len(dates_sorted) + 1):
        subset = df[df[date_col].isin(dates_sorted[:i])]
        c_users = subset[subset[group_col] == 'control'].groupby(user_col)[metric_col].sum()
        t_users = subset[subset[group_col] == 'treatment'].groupby(user_col)[metric_col].sum()
        
        # Pad with zeros for non-purchasing users
        c_all = pd.Series(0.0, index=range(n_control))
        t_all = pd.Series(0.0, index=range(n_control, n_control + n_treatment))
        c_all.update(c_users)
        t_all.update(t_users)
        
        diff = t_all.mean() - c_all.mean()
        se = np.sqrt(c_all.var()/n_control + t_all.var()/n_treatment)
        lift = diff / c_all.mean() if c_all.mean() > 0 else 0
        lift_se = se / c_all.mean() if c_all.mean() > 0 else 0
        
        cum_lifts.append(lift * 100)
        cum_ci_low.append((lift - 1.96 * lift_se) * 100)
        cum_ci_high.append((lift + 1.96 * lift_se) * 100)
    
    axes[0, 1].plot(dates_sorted, cum_lifts, 'g-', linewidth=2)
    axes[0, 1].fill_between(dates_sorted, cum_ci_low, cum_ci_high, alpha=0.2, color='green')
    axes[0, 1].axhline(0, color='black', linewidth=0.5)
    axes[0, 1].axhline(5, color='orange', linestyle='--', alpha=0.7, label='True effect')
    axes[0, 1].set_title('Cumulative Lift with 95% CI')
    axes[0, 1].set_ylabel('Lift (%)')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # --- Panel 3: Sample ratio check ---
    daily_n_c = df[df[group_col] == 'control'].groupby(date_col)[user_col].nunique()
    daily_n_t = df[df[group_col] == 'treatment'].groupby(date_col)[user_col].nunique()
    ratio = daily_n_t / (daily_n_c + daily_n_t)
    
    axes[1, 0].plot(ratio.index, ratio.values, 'purple', linewidth=1.5)
    axes[1, 0].axhline(0.5, color='black', linestyle='--', alpha=0.5)
    axes[1, 0].axhspan(0.49, 0.51, alpha=0.1, color='green', label='Acceptable range')
    axes[1, 0].set_title('Sample Ratio Mismatch Check')
    axes[1, 0].set_ylabel('Treatment fraction')
    axes[1, 0].set_ylim([0.45, 0.55])
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # --- Panel 4: Rolling 7-day lift ---
    if len(dates_sorted) >= 7:
        rolling_lifts = []
        rolling_dates = []
        for i in range(6, len(dates_sorted)):
            week_dates = dates_sorted[i-6:i+1]
            subset = df[df[date_col].isin(week_dates)]
            c_rev = subset[subset[group_col] == 'control'][metric_col].sum() / n_control
            t_rev = subset[subset[group_col] == 'treatment'][metric_col].sum() / n_treatment
            rolling_lifts.append((t_rev - c_rev) / c_rev * 100 if c_rev > 0 else 0)
            rolling_dates.append(dates_sorted[i])
        
        axes[1, 1].plot(rolling_dates, rolling_lifts, 'darkorange', linewidth=2)
        axes[1, 1].axhline(0, color='black', linewidth=0.5)
        axes[1, 1].axhline(5, color='green', linestyle='--', alpha=0.7, label='True effect')
        axes[1, 1].set_title('Rolling 7-Day Lift')
        axes[1, 1].set_ylabel('Lift (%)')
        axes[1, 1].legend()
        axes[1, 1].grid(alpha=0.3)
        axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.suptitle('QuickCart Experiment Dashboard: New Checkout Flow', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


experiment_dashboard(df)

:::{admonition} Reading the Dashboard
:class: note

**Panel 1 (Time Series)**: Both groups show the same seasonality pattern (weekends up). Treatment is consistently above control -- good sign.

**Panel 2 (Cumulative Lift)**: The lift starts high (novelty effect), then stabilizes. The CI narrows over time as we accumulate more data. Once the CI excludes zero, we have statistical significance.

**Panel 3 (Sample Ratio)**: The treatment fraction should hover around 0.50. Large deviations indicate a randomization bug (a critical issue that invalidates the experiment).

**Panel 4 (Rolling Lift)**: A 7-day rolling window smooths out daily noise while revealing trends. If the lift is declining over time, it might be novelty wearing off.
:::

---

## 8. Anomaly Detection in Metric Time Series

Even in a well-run experiment, things go wrong: server outages, marketing campaigns, holidays, data pipeline failures. We need automated anomaly detection to flag days that need investigation.

### Approach: Moving Z-Score

For each day, compute how many standard deviations the metric is from its recent history:

$$
z_t = \frac{x_t - \bar{x}_{t-w:t-1}}{s_{t-w:t-1}}
$$

where $w$ is the lookback window (typically 7 days to capture a full weekly cycle).

In [ ]:
def detect_anomalies(series, window=7, threshold=2.5):
    """
    Detect anomalies using a moving z-score approach.
    
    Parameters
    ----------
    series : array-like
        Time series of metric values.
    window : int
        Lookback window for computing mean and std.
    threshold : float
        Z-score threshold for flagging anomalies.
    
    Returns
    -------
    z_scores : np.array
        Z-score for each point (NaN for insufficient history).
    anomalies : np.array of bool
        True for anomalous points.
    """
    series = np.array(series, dtype=float)
    n = len(series)
    z_scores = np.full(n, np.nan)
    
    for i in range(window, n):
        lookback = series[i-window:i]
        mu = lookback.mean()
        sigma = lookback.std()
        if sigma > 0:
            z_scores[i] = (series[i] - mu) / sigma
        else:
            z_scores[i] = 0
    
    anomalies = np.abs(z_scores) > threshold
    return z_scores, anomalies


# Create a longer dataset with injected anomalies for demonstration
np.random.seed(123)
n_days_demo = 60
base_revenue = 50000
daily_revenue_demo = np.array([
    base_revenue * (1.3 if i % 7 >= 5 else 1.0) * np.random.normal(1, 0.08)
    for i in range(n_days_demo)
])

# Inject anomalies
daily_revenue_demo[15] *= 0.4   # Server outage (day 16)
daily_revenue_demo[35] *= 1.8   # Flash sale (day 36)
daily_revenue_demo[50] *= 0.5   # Data pipeline failure (day 51)

# Detect
z_scores, anomalies = detect_anomalies(daily_revenue_demo, window=7, threshold=2.5)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
demo_dates = pd.date_range('2024-01-01', periods=n_days_demo)

axes[0].plot(demo_dates, daily_revenue_demo, 'b-', linewidth=1)
axes[0].scatter(demo_dates[anomalies], daily_revenue_demo[anomalies], 
               color='red', s=100, zorder=5, label='Anomaly detected')
axes[0].set_ylabel('Daily Revenue ($)')
axes[0].set_title('Metric Time Series with Anomaly Detection')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(demo_dates, z_scores, 'purple', linewidth=1)
axes[1].axhline(2.5, color='red', linestyle='--', alpha=0.7, label='Threshold (+/-2.5)')
axes[1].axhline(-2.5, color='red', linestyle='--', alpha=0.7)
axes[1].scatter(demo_dates[anomalies], z_scores[anomalies], 
               color='red', s=100, zorder=5)
axes[1].set_ylabel('Z-score')
axes[1].set_xlabel('Date')
axes[1].set_title('Moving Z-Score (window=7 days)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

anomaly_dates = demo_dates[anomalies]
print(f"Anomalies detected on {len(anomaly_dates)} days:")
for d, z in zip(anomaly_dates, z_scores[anomalies]):
    print(f"  {d.strftime('%Y-%m-%d')}: z-score = {z:.2f}")

### What to Do When Anomalies Are Detected

| Anomaly Type | Action | Impact on Experiment |
|-------------|--------|---------------------|
| **Server outage** | Exclude affected days | Both groups affected equally -- usually safe to exclude |
| **Data pipeline failure** | Investigate and backfill | May need to extend experiment |
| **Marketing campaign** | Check if it targeted one group | Can invalidate experiment if unbalanced |
| **Holiday/event** | Include but document | Seasonality is real user behavior |

:::{admonition} Advanced: Day-of-Week Aware Anomaly Detection
:class: dropdown

The simple moving z-score doesn't account for day-of-week patterns. A Saturday will often look anomalous compared to the preceding Monday-Friday window.

Better approaches:
1. **Same-day comparison**: Compare this Saturday to previous Saturdays
2. **Seasonal decomposition**: Remove the weekly pattern, detect anomalies in the residual
3. **STL decomposition**: `statsmodels.tsa.seasonal.seasonal_decompose`

```python
# Same-day-of-week z-score
def dow_anomaly_detection(series, dates, n_weeks_lookback=4, threshold=2.5):
    anomalies = []
    for i, (val, date) in enumerate(zip(series, dates)):
        # Get same day-of-week from previous weeks
        same_dow = [series[j] for j in range(i-7, -1, -7) 
                    if j >= 0][:n_weeks_lookback]
        if len(same_dow) >= 2:
            mu, sigma = np.mean(same_dow), np.std(same_dow)
            if sigma > 0 and abs(val - mu) / sigma > threshold:
                anomalies.append(i)
    return anomalies
```
:::

---

## 9. Putting It All Together: A Complete Analysis Pipeline

Let's build a complete analysis function that takes raw experiment data and produces a decision-ready summary.

In [ ]:
def analyze_experiment(df, group_col='group', date_col='day', 
                       metric_col='cost', user_col='user_id',
                       n_control=2500, n_treatment=2500,
                       burn_in_days=3, alpha=0.05):
    """
    Complete experiment analysis pipeline.
    
    Steps:
    1. Check for sample ratio mismatch
    2. Remove burn-in period (novelty)
    3. Check for anomalous days
    4. Compute per-user metrics
    5. Run statistical tests
    6. Report results with confidence intervals
    """
    print("=" * 60)
    print("EXPERIMENT ANALYSIS REPORT")
    print("=" * 60)
    
    # Step 1: Sample ratio check
    n_c = df[df[group_col] == 'control'][user_col].nunique()
    n_t = df[df[group_col] == 'treatment'][user_col].nunique()
    ratio = n_t / (n_c + n_t)
    chi2_stat = (n_c - n_t)**2 / (n_c + n_t)
    srm_pvalue = 1 - stats.chi2.cdf(chi2_stat, df=1)
    
    print(f"\n[1] SAMPLE RATIO CHECK")
    print(f"    Control users: {n_c}, Treatment users: {n_t}")
    print(f"    Ratio: {ratio:.4f} (expected: 0.5000)")
    print(f"    SRM p-value: {srm_pvalue:.4f}")
    if srm_pvalue < 0.001:
        print("    *** WARNING: Sample Ratio Mismatch detected! ***")
    else:
        print("    Status: OK")
    
    # Step 2: Remove burn-in
    all_dates = sorted(df[date_col].unique())
    post_burnin_dates = all_dates[burn_in_days:]
    df_clean = df[df[date_col].isin(post_burnin_dates)]
    
    print(f"\n[2] BURN-IN REMOVAL")
    print(f"    Removed first {burn_in_days} days (novelty period)")
    print(f"    Analysis period: {post_burnin_dates[0]} to {post_burnin_dates[-1]}")
    print(f"    Days in analysis: {len(post_burnin_dates)}")
    
    # Step 3: Anomaly check
    daily_total = df_clean.groupby(date_col)[metric_col].sum().values
    _, anomaly_flags = detect_anomalies(daily_total, window=7, threshold=3.0)
    n_anomalies = anomaly_flags.sum()
    
    print(f"\n[3] ANOMALY CHECK")
    print(f"    Anomalous days detected: {n_anomalies}")
    if n_anomalies > 0:
        print("    (Not excluded -- investigate manually)")
    else:
        print("    Status: Clean")
    
    # Step 4: Per-user metrics
    user_revenue_c = df_clean[df_clean[group_col] == 'control'].groupby(user_col)[metric_col].sum()
    user_revenue_t = df_clean[df_clean[group_col] == 'treatment'].groupby(user_col)[metric_col].sum()
    
    # Pad with zeros for users who didn't purchase
    all_c = pd.Series(0.0, index=range(n_control))
    all_t = pd.Series(0.0, index=range(n_control, n_control + n_treatment))
    all_c.update(user_revenue_c)
    all_t.update(user_revenue_t)
    
    # Step 5: Statistical test
    t_stat, p_value = stats.ttest_ind(all_c.values, all_t.values, equal_var=False)
    
    mean_c = all_c.mean()
    mean_t = all_t.mean()
    diff = mean_t - mean_c
    se = np.sqrt(all_c.var()/n_control + all_t.var()/n_treatment)
    ci_low = diff - 1.96 * se
    ci_high = diff + 1.96 * se
    
    lift = diff / mean_c
    lift_ci_low = ci_low / mean_c
    lift_ci_high = ci_high / mean_c
    
    print(f"\n[4] RESULTS")
    print(f"    Control mean:     ${mean_c:.2f} per user")
    print(f"    Treatment mean:   ${mean_t:.2f} per user")
    print(f"    Absolute diff:    ${diff:.2f} [{ci_low:.2f}, {ci_high:.2f}]")
    print(f"    Relative lift:    {lift:+.2%} [{lift_ci_low:+.2%}, {lift_ci_high:+.2%}]")
    print(f"    p-value:          {p_value:.4f}")
    
    # Step 6: Decision
    print(f"\n[5] DECISION")
    if p_value < alpha and lift > 0:
        print(f"    SHIP IT -- Statistically significant positive effect.")
    elif p_value < alpha and lift < 0:
        print(f"    REVERT -- Statistically significant negative effect.")
    else:
        print(f"    INCONCLUSIVE -- Cannot reject the null at alpha={alpha}.")
    
    print("=" * 60)
    
    return {
        'lift': lift, 'ci': (lift_ci_low, lift_ci_high),
        'p_value': p_value, 'srm_ok': srm_pvalue >= 0.001
    }


results = analyze_experiment(df)

---

## 10. Effect of Analysis Period on Results

A critical lesson: **when** you look at results matters. Let's show how the measured effect changes depending on which subset of days we analyze.

In [ ]:
# How does the measured effect change with different analysis windows?
periods = [3, 5, 7, 10, 14, 21, 28]
period_results = []

for period in periods:
    metrics_c = calculate_sales_metrics(df, period=period, filters={'group': 'control'})
    metrics_t = calculate_sales_metrics(df, period=period, filters={'group': 'treatment'})
    lift = (metrics_t['avg_daily_revenue'] - metrics_c['avg_daily_revenue']) / metrics_c['avg_daily_revenue']
    period_results.append({'period': period, 'lift': lift * 100})

period_df = pd.DataFrame(period_results)

plt.figure(figsize=(10, 5))
plt.bar(period_df['period'].astype(str), period_df['lift'], 
        color='steelblue', alpha=0.8, edgecolor='navy')
plt.axhline(5, color='green', linestyle='--', linewidth=2, label='True effect (5%)')
plt.xlabel('Analysis Period (last N days)')
plt.ylabel('Measured Lift (%)')
plt.title('Measured Treatment Effect by Analysis Window')
plt.legend()
plt.grid(alpha=0.3, axis='y')
plt.show()

print("Lesson: Shorter windows are noisier and may include novelty effects.")
print("Always use complete weeks (7, 14, 21, 28 days) to avoid day-of-week bias.")

---

## Summary

| Topic | Key Takeaway |
|-------|-------------|
| Event-to-metric | Raw events must be aggregated to user or day level before analysis |
| `calculate_sales_metrics` | Filter, aggregate daily, compute summary -- the core pattern |
| Metric definition | Define primary, secondary, and guardrail metrics **before** the experiment |
| Good metric properties | Sensitive, directional, attributable, timely, robust |
| Daily monitoring | Always plot time series -- single numbers hide problems |
| Day-of-week | Analyze in complete weeks to avoid seasonality bias |
| Novelty effects | First few days often show inflated effects; use burn-in removal |
| Metric sensitivity | Binary metrics (conversion) detect effects faster than continuous (revenue) |
| Dashboard | Four panels: time series, cumulative lift with CI, SRM check, rolling lift |
| Anomaly detection | Moving z-score flags days needing investigation |
| Analysis pipeline | Systematic: SRM check, burn-in, anomalies, test, decide |

### What's Next

In **Week 3**, we'll dive into sample size calculation and minimum detectable effects -- answering the question "How long should we run this experiment?" before it starts. This connects directly to the metric sensitivity analysis we did here.